In [1]:
import pandas as pd
import sys
sys.path.append('../../')
from MEGA_utilities import data_col_standardize, data_remove_duplicate


# load data
Crick_H3N2 = pd.read_excel('../../../data/raw/data4model(Crick-H1N1).xlsx')
origin_df = Crick_H3N2.copy()

## DNA MEGA

In [2]:
## select required columns
H1N1_data_filt1 = origin_df[['serumName','virusName','serumHA', 'serumNA', 'virusHA', 'virusNA', 
                           'serumPassCat','virusPassCat', 'serumType','HI_Dist']].copy()
## remove duplicated row and mean HI_Dist
H1N1_data_filt2 = H1N1_data_filt1.groupby(['serumHA', 'serumNA', 'virusHA', 'virusNA', 'serumPassCat', 'virusPassCat']) \
        .agg({'serumName': 'first', 'virusName': 'first', 'serumType': 'first', 'HI_Dist': 'mean'}) \
        .reset_index()[['serumName', 'virusName', 'serumHA', 'serumNA', 'virusHA', 'virusNA',
                        'serumPassCat', 'virusPassCat', 'serumType', 'HI_Dist']]
## remove PassCat = 'BOTH'
H1N1_data_filt3 = H1N1_data_filt2[(H1N1_data_filt2['serumPassCat'] != 'BOTH') &
                                  (H1N1_data_filt2['virusPassCat'] != 'BOTH')].reset_index(drop=True)
## replace PassCat to special token
H1N1_data_filt4 = H1N1_data_filt3.replace({'serumPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'},
                                           'virusPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'}})

H1N1_data_final = H1N1_data_filt4.copy()

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader

class AADataset(Dataset):
    def __init__(self, DataFrame):
        self.sequence = (DataFrame['serumHA'] + '<eos>' + DataFrame['serumNA'] + '<eos>' + DataFrame['virusHA'] + \
                         '<eos>' + DataFrame['virusNA'] + '<eos>' + DataFrame['serumType'] + \
                         '<eos>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat']).tolist()
        self.labels = torch.tensor(DataFrame['HI_Dist'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequence[idx], self.labels[idx]

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(H1N1_data_final, test_size=0.1, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=1/9, random_state=42)

train_dataset = AADataset(train_df)
valid_dataset = AADataset(valid_df)
test_dataset = AADataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [6]:
from bio_tokenizer import BioTokenizer
from transformers import MegaConfig, MegaForSequenceClassification
from MEGA_utilities import count_parameters
from torch.optim import AdamW
from transformers import get_scheduler
import torch

# get tokenizer and model
tokenizer = BioTokenizer(vocab_file='./vocab_AA.txt')

# update the num_vocab and num_label
config = MegaConfig()
config.num_labels=1
config.vocab_size=28
config.max_positions=4000
config.num_attention_heads=4
config.num_hidden_layers=5
device = torch.device("cuda:0")
model = MegaForSequenceClassification(config)
model.to(device)
print("Number of parameters: %e"%count_parameters(model))

# optimizer
optimizer = AdamW(model.parameters(), lr=1e-4)

# scheduler
num_epochs = 160
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(name="linear", optimizer=optimizer,
                             num_warmup_steps=len(train_loader), num_training_steps=num_training_steps)

Number of parameters: 1.135435e+06


In [7]:
from tqdm import tqdm
from utilities import print_exams
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from utilities import EarlyStopping
from datetime import datetime

progress_bar = tqdm(range(num_training_steps))
early_stopping = EarlyStopping(patience=10, delta=0.005, save_dir='./1.3_H1N1_only_model/')

# Training loop
for epoch in range(num_epochs):
    model.train()
    loss_ls = []
    for batch_seq, batch_label in train_loader:
        batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
        batch_input = batch_input.to(device)
        batch_label = batch_label.to(device)

        outputs = model(**batch_input, labels=batch_label)

        loss = outputs.loss
        loss_ls.append(loss.item())

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    train_loss = sum(loss_ls) / len(loss_ls)
    print('train loss :', train_loss)

    prediction_ls = []
    reference_ls = []
    logits_ls = []
    loss_ls_valid = []
    with torch.no_grad():
        model.eval()
        for batch_seq, batch_label in valid_loader:
            batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
            batch_input = batch_input.to(device)
            batch_label = batch_label.to(device)

            outputs = model(**batch_input, labels=batch_label)
            logits = outputs.logits
            loss = outputs.loss

            logits_ls.append(logits)
            loss_ls_valid.append(loss.item())
            prediction_ls += logits.tolist()
            prediction_ls_final = []
            for sublist in prediction_ls:
                for element in sublist:
                    prediction_ls_final.append(element)
            reference_ls += batch_label.tolist()

    print_exams(prediction_ls_final, reference_ls)
    valid_MAE = mean_absolute_error(reference_ls, prediction_ls_final)
    valid_mse = mean_squared_error(reference_ls, prediction_ls_final)
    valid_pearson = pearsonr(reference_ls, prediction_ls_final).statistic
    valid_spearman = spearmanr(reference_ls, prediction_ls_final).statistic
    
    early_stopping(valid_mse, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

    ## 将epoch信息写入log.txt
    with open('./1.3_H1N1_only_model/log.txt', 'a') as f:
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"[{current_time}] Epoch {epoch + 1}/{num_epochs}, train loss: {train_loss:.4f}, valid MAE: {valid_MAE:.4f}, valid MSE: {valid_mse:.4f}, valid Pearson: {valid_pearson:.4f}, valid Spearman: {valid_spearman:.4f}\n")

  1%|          | 3002/480480 [05:31<14:37:47,  9.07it/s]

train loss : 2.8937854501076075


  1%|          | 3004/480480 [06:04<1021:42:23,  7.70s/it]

MAE:  1.17871080492612
MSE:  2.6316311883243757
pearson correlation:  PearsonRResult(statistic=0.3643958637229695, pvalue=3.2352243488659054e-118)
spearman correlation:  SignificanceResult(statistic=0.3249307588575627, pvalue=5.225891686437417e-93)
Validation MSE decrease (inf --> 2.631631).  Saving model ...


  1%|          | 6005/480480 [11:34<14:30:02,  9.09it/s]  

...

train loss : 0.6343926843696144


 77%|███████▋  | 369370/480480 [12:26:55<236:36:19,  7.67s/it]

MAE:  0.6540438443454081
MSE:  0.7050887297917146
pearson correlation:  PearsonRResult(statistic=0.8775718568896771, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.7337305422036922, pvalue=0.0)
EarlyStopping counter: 9 out of 10


 77%|███████▋  | 372371/480480 [12:32:25<3:18:33,  9.07it/s]  

train loss : 0.630951145086394
MAE:  0.6514982312199055
MSE:  0.7058630728979643
pearson correlation:  PearsonRResult(statistic=0.8766175637131892, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.7323251799618086, pvalue=0.0)
EarlyStopping counter: 10 out of 10
Early stopping
